# Online Retail Analysis

This notebook walks through a beginner-friendly retail data analysis project using **pandas** and **NumPy**.

## Steps covered
1. Load the data
2. Inspect the dataset
3. Handle missing values
4. Remove duplicates
5. Create new calculated columns
6. Perform exploratory data analysis
7. Generate insights

In [ ]:
import pandas as pd
import numpy as np

## Load the dataset

In [ ]:
df = pd.read_csv("../data/sales_data.csv")
df.head()

## Understand the dataset

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

## Handle missing values

In [ ]:
df["discount"] = df["discount"].fillna(df["discount"].mean())
df.isnull().sum()

## Remove duplicate rows

In [ ]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)

print("Rows before removing duplicates:", before_rows)
print("Rows after removing duplicates:", after_rows)
print("Duplicates removed:", before_rows - after_rows)

## Convert data types

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df.dtypes

## Create transformed columns

In [ ]:
df["revenue"] = df["quantity"] * df["price"]
df["discount_amount"] = df["revenue"] * df["discount"]
df["final_revenue"] = df["revenue"] - df["discount_amount"]
df["month"] = df["order_date"].dt.to_period("M").astype(str)

df["segment"] = np.where(
    df["final_revenue"] > 500,
    "High",
    np.where(df["final_revenue"] > 200, "Medium", "Low")
)

df.head()

## Exploratory Data Analysis

In [ ]:
category_analysis = (
    df.groupby("category", as_index=False)
      .agg(total_orders=("order_id", "count"),
           total_quantity=("quantity", "sum"),
           total_final_revenue=("final_revenue", "sum"))
      .sort_values("total_final_revenue", ascending=False)
)
category_analysis

In [ ]:
region_analysis = (
    df.groupby("region", as_index=False)
      .agg(total_orders=("order_id", "count"),
           total_final_revenue=("final_revenue", "sum"))
      .sort_values("total_final_revenue", ascending=False)
)
region_analysis

In [ ]:
monthly_trend = (
    df.groupby("month", as_index=False)
      .agg(total_final_revenue=("final_revenue", "sum"))
      .sort_values("month")
)
monthly_trend

In [ ]:
segment_analysis = (
    df.groupby("segment", as_index=False)
      .agg(customer_count=("customer_id", "nunique"),
           total_final_revenue=("final_revenue", "sum"))
      .sort_values("total_final_revenue", ascending=False)
)
segment_analysis

In [ ]:
pivot_table = pd.pivot_table(
    df,
    values="final_revenue",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0
)
pivot_table

## Key Insights

- Electronics is one of the highest revenue-generating categories.
- East and South regions contribute strong final revenue.
- High-value orders make a large contribution even with fewer transactions.
- Missing discounts were handled using the average discount to keep the dataset consistent.
- Duplicate rows were removed before final analysis to avoid overstating revenue.